# 4.1 — Naive (lower bound)

> Run `4.0_cl_setup.ipynb` first.

**Mechanism:** none. Train on each new task, do nothing to protect the old ones.

**Memory:** 0 bytes.

## Why this arm exists

It is the **lower bound**. Base accuracy should collapse — if it doesn't, forgetting
isn't happening and something is wrong with the label space or the protocol.

Every other arm is measured as "how much of that collapse did we recover".

## Setup

In [ ]:
# ── ULAL bootstrap — self-contained, no Drive dependency ────────────────────
# One shallow git clone instead of hundreds of API calls. subprocess takes a
# LIST, never a shell string, so an '&' in a branch name is safe.
import os, sys, subprocess
from getpass import getpass

REPO   = "silvergjeka22/Unified-Lifelong-Action-Learning"
BRANCH = "embeddings"
ROOT   = "/content/ulal"

token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token (hidden): ")
os.environ["GITHUB_TOKEN"] = token

if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://" + token + "@github.com/" + REPO + ".git", ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

print(subprocess.run(["git", "-C", ROOT, "log", "-1", "--oneline"],
                     capture_output=True, text=True).stdout.strip())

# ── Verify the clone contains what this notebook needs ──────────────────────
# The clone reflects what is ON GITHUB. Local commits you never pushed are
# invisible here, and the symptom is a confusing ModuleNotFoundError later.
_need = ["src/__init__.py", "src/bootstrap.py", "src/imports.py",
         "src/data/cache.py", "src/cl/trainer.py",
         "src/models/temporal_head.py", "src/utils/probe.py"]
_miss = [f for f in _need if not os.path.exists(os.path.join(ROOT, f))]

if _miss:
    print("ERROR — the cloned repo is missing:")
    for f in _miss:
        print("   ", f)
    print("")
    print("These exist locally but are not on the '" + BRANCH + "' branch on GitHub.")
    print("On your machine run:")
    print("    git add -A")
    print("    git commit -m 'sync src'")
    print("    git push origin " + BRANCH)
    print("then re-run this cell.")
    raise RuntimeError("src/ incomplete on GitHub — push your commits first.")

print("src/ complete (" + str(len(_need)) + " key files present)")

In [ ]:
# ── Drive + dataset + paths ─────────────────────────────────────────────────
# Exports ULAL_* env vars that config.py reads. Nothing is written to disk and
# no kernel restart is needed.
os.environ["KAGGLE_USERNAME"] = "YOUR_KAGGLE_USER"
os.environ["KAGGLE_KEY"]      = "YOUR_KAGGLE_KEY"

from src.bootstrap import setup
setup(drive=True, dataset=True)

In [ ]:
%run /content/ulal/src/imports.py

In [ ]:
# ── Global label space (src/data/cache.py) ──────────────────────────────────
TASK_CLASSES = [cfg.SELECTED_CLASSES, cfg.TASK_1, cfg.TASK_2, cfg.TASK_4]
TASK_ROOTS   = [cfg.BASE_ROOT, cfg.TASK1_ROOT, cfg.TASK2_ROOT, cfg.TASK4_ROOT]
TASK_NAMES   = ["Base", "Task1", "Task2", "Task4"]
CL_TASKS     = [1, 2]     # Sections 4-5 stream over these
FEWSHOT_TASK = 3          # Section 7 probe only

global_classes, global_c2i, NUM_AFTER = build_label_space(TASK_CLASSES)
describe_label_space(TASK_CLASSES, TASK_NAMES, NUM_AFTER, task_3=cfg.TASK_3)

In [ ]:
# ── Load cached frame features (src/data/cache.py) ──────────────────────────
CACHE, LSTM_STATE, META = load_feature_cache(cfg.FEAT_CACHE)

In [ ]:
# ── Shared protocol (written by 4.0_cl_setup) ───────────────────────────────
# Every arm loads the SAME base head and the SAME hyperparameters, so lr/epochs/
# schedule are identical by construction even though the arms run in separate
# notebooks. The mechanism flags are the only difference.
P = json.load(open(f"{cfg.RESULTS_DIR}/stage1_protocol.json"))

FREEZE_LSTM = P["freeze_lstm"]
CEILING     = P["ceiling"]
CL_EPOCHS   = P["epochs"]
CL_LR       = P["lr"]
CL_WD       = P["wd"]
CL_BATCH    = P["batch_size"]
CL_LS       = P["label_smoothing"]

base_head = make_temporal_head(NUM_AFTER[0], freeze_lstm=FREEZE_LSTM, device=device)
base_head.load_state_dict(torch.load(f"{cfg.CKPT_DIR}/head_base.pt", map_location=device))
base_head.eval()

STREAM = dict(
    base_head=base_head, cache=CACHE, cl_tasks=CL_TASKS, task_classes=TASK_CLASSES,
    task_names=TASK_NAMES, num_after=NUM_AFTER, global_c2i=global_c2i,
    base_classes=cfg.SELECTED_CLASSES, device=device,
    epochs=CL_EPOCHS, lr=CL_LR, wd=CL_WD, batch_size=CL_BATCH,
    label_smoothing=CL_LS,
)

print(f"Protocol : {CL_EPOCHS} epochs, lr={CL_LR}, batch={CL_BATCH}, no best-val selection")
print(f"Boundary : LSTM {'FROZEN' if FREEZE_LSTM else 'TRAINABLE'}")
print(f"Ceiling  : {CEILING:.2%}" if CEILING else "Ceiling  : (none — run Section 3)")
print(f"Base head: {base_head.trainable_params():,} trainable, "
      f"base acc {eval_tasks_upto(base_head, CACHE, 0, device)[0]:.2%}")

## Run the arm

In [ ]:
res = run_cl_stream(arm="naive", **STREAM)

R, m = save_arm_result(
    f"{cfg.RESULTS_DIR}/stage1_arm_naive.json",
    "Naive", res["rows"], TASK_NAMES, res["bytes_per_class"],
    config={"mechanism": "none"}, ceiling=CEILING,
)
cl_report(R, "Naive", [TASK_NAMES[i] for i in [0] + CL_TASKS])
torch.save(res["head"].state_dict(), f"{cfg.CKPT_DIR}/head_naive.pt")

---

**Expected:** a sharp drop in base accuracy after each task, and high accuracy on the
task just learned. That gap is catastrophic forgetting.

**If base accuracy does NOT drop**, stop and check: are the task label spaces disjoint?
Is the head actually expanding? Run 4.7 only after every arm has run.